# Course 6: From Beginner to an AI Research Lab
## Session 8: Reproducing a paper

**Goal:** plan a reproduction that fits your compute, run the sanity checks before the real run, and keep an honest deviations log.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026). Cells labelled NOT RUN need a GPU, a download or an account. Cells that are worksheets hold example rows for you to replace with your own.

### 1. The plan (worksheet)

In [ ]:
plan = {
    "paper": "<title, venue, year>",
    "claim to reproduce": "<one sentence with the number, e.g. 'Table 2: 71.3% accuracy on X with model Y'>",
    "metric and how it is computed": "",
    "data": "",
    "what I will re-implement myself": "data loading, model, training loop, evaluation",
    "what I will take from the official code": "only details the paper does not state",
    "compute available": "<e.g. 30 GPU hours a week on a T4>",
    "stop-and-write-up date": "<two weeks from today>",
}
for k, v in plan.items(): print(f"{k:42s} {v}")

### 2. Will it fit? Time one short run
Train for 2% of the planned steps, multiply up, and check that five seeds plus a baseline and an ablation fit your quota. If not, shrink something now and write it in the deviations log.

In [ ]:
def budget(seconds_for_short_run, fraction_run=0.02, runs_needed=7, quota_hours=30, overhead=1.2):
    """runs_needed: 5 seeds of the main result + 1 baseline + 1 ablation, as a minimum."""
    hours_per_run = seconds_for_short_run / fraction_run / 3600 * overhead
    total = hours_per_run * runs_needed
    return {"hours per full run": round(hours_per_run, 2), "hours for all runs": round(total, 1), "quota": quota_hours, "fits": total <= quota_hours,
            "scale steps or data by": 1.0 if total <= quota_hours else round(quota_hours / total, 2)}
print(budget(seconds_for_short_run=180)); print(budget(seconds_for_short_run=900))
assert budget(180)["fits"] and not budget(900)["fits"]

### 3. Sanity checks before the real run
Three checks, demonstrated on a small stand-in task. Use the same functions on your reproduction.
1. The loss at initialization is what theory says (`ln(number of classes)` for a classifier).
2. The model can overfit one small batch.
3. Your evaluation code gives the published number when fed a released checkpoint. Here that is imitated by scoring fixed predictions with a known answer.

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from sklearn.datasets import load_digits
torch.manual_seed(0)
d = load_digits(); X = torch.tensor(d.data / 16.0, dtype=torch.float32); y = torch.tensor(d.target)
make_model = lambda: nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 10))

def check_init_loss(model, X, y, n_classes, tol=0.2):
    loss = F.cross_entropy(model(X), y).item(); return abs(loss - np.log(n_classes)) < tol, round(loss, 3)
def check_overfit_one_batch(model, X, y, steps=300, lr=1e-2, target=0.01):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(steps): loss = F.cross_entropy(model(X), y); opt.zero_grad(); loss.backward(); opt.step()
    return loss.item() < target, round(loss.item(), 5)
def accuracy_metric(pred, target): return float((pred == target).float().mean())
def check_metric():
    pred, target = torch.tensor([1, 2, 3, 4, 0, 0, 0, 0]), torch.tensor([1, 2, 3, 4, 1, 2, 3, 4]); return accuracy_metric(pred, target) == 0.5

print("1. loss at initialization:", check_init_loss(make_model(), X, y, 10))
print("2. overfit one batch of 8:", check_overfit_one_batch(make_model(), X[:8], y[:8]))
print("3. metric returns the known answer:", check_metric())
assert check_init_loss(make_model(), X, y, 10)[0] and check_overfit_one_batch(make_model(), X[:8], y[:8])[0] and check_metric()

### 4. One run, then compare with the paper

In [ ]:
PAPER_NUMBER = None            # TODO: the number from the paper, with its unit
MY_NUMBER = None               # TODO: your one-seed result
def compare(paper, mine):
    if paper is None or mine is None: return "fill in both numbers"
    gap = mine - paper; return f"paper {paper} | mine {mine} | gap {gap:+.2f} ({100 * gap / paper:+.1f}%). One seed: this is a starting point, not a verdict."
print(compare(PAPER_NUMBER, MY_NUMBER)); print(compare(71.3, 69.8))

### 5. The deviations log
Every difference you know of goes in, however small. Start it on day one, not when the number disappoints.

In [ ]:
import pandas as pd
deviations = pd.DataFrame([
    {"deviation_from_the_paper": "Batch size 64, not 256", "why": "T4 memory", "likely_effect": "small, more noise"},
    {"deviation_from_the_paper": "20k steps, not 100k", "why": "weekly GPU quota", "likely_effect": "lower accuracy"},
    {"deviation_from_the_paper": "Augmentation details guessed", "why": "not stated in paper", "likely_effect": "unknown"},
])
deviations.to_csv("deviations.csv", index=False); print(deviations.to_string(index=False))

### 6. When the number does not match: a fixed order of checks
1. Evaluation: are you computing the metric the way the paper does (which split, which averaging)?
2. Data: same version, same preprocessing, same split sizes?
3. Training length and schedule: compare at the paper's number of steps, or say that you could not.
4. Model: count the parameters and compare with the paper's figure.
5. Seeds: is the gap larger than the spread across seeds? (Session 9.)

Change one thing between runs and log every run.

In [ ]:
def count_parameters(model): return sum(p.numel() for p in model.parameters())
print("parameters in the stand-in model:", count_parameters(make_model()), "(compare your count with the paper's before you compare accuracies)")

### 7. The two-page report
The claim, your setup, your number beside the paper's, the deviations, the compute used and what you learned. A careful report of a result that did not reproduce is a contribution. A matching number with no log of deviations is not.

### TODO
1. Fill in the plan and the budget for your paper.
2. Re-implement piece by piece, run the three checks, then one seed.
3. Hand in the draft report with the deviations log.